# 01 - Data Exploration

Explore the CSE stock data, 25 technical indicators, and walk-forward validation splits.
No training happens here - this notebook is for understanding and presenting the dataset.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from preprocessor import load_ticker, get_folds, FEATURE_COLS, TARGET_COL
from utils.visualizer import plot_fold_splits, plot_indicator_correlation

TICKERS = [
    'JKH.N0000', 'COMB.N0000', 'DIAL.N0000', 'HNB.N0000',  'LOLC.N0000',
    'SAMP.N0000', 'NTB.N0000', 'HHL.N0000',  'DIST.N0000', 'HAYL.N0000',
]

print('Imports OK')

## 1. Dataset Overview

In [ ]:
rows = []
for ticker in TICKERS:
    df = load_ticker(ticker)
    rows.append({
        'Ticker':     ticker.replace('.N0000', ''),
        'Rows':       len(df),
        'Start':      str(df.index[0].date()),
        'End':        str(df.index[-1].date()),
        'Min Close':  f"{df[TARGET_COL].min():.2f}",
        'Max Close':  f"{df[TARGET_COL].max():.2f}",
        'Mean Close': f"{df[TARGET_COL].mean():.2f}",
    })

pd.DataFrame(rows).set_index('Ticker')

## 2. Close Price - All 10 Tickers

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 14))
axes = axes.flatten()

for ax, ticker in zip(axes, TICKERS):
    df = load_ticker(ticker)
    ax.plot(df.index, df[TARGET_COL], linewidth=0.8, color='#2c3e50')
    ax.set_title(ticker.replace('.N0000', ''), fontsize=11)
    ax.set_ylabel('Close (LKR)', fontsize=8)
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('CSE Stock Close Prices - 2021 to 2025', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 3. Walk-Forward Validation Splits - JKH

In [ ]:
df_jkh = load_ticker('JKH.N0000')
path = plot_fold_splits(df_jkh, 'JKH.N0000')

from IPython.display import Image
Image(path)

## 4. Walk-Forward Splits - All Tickers

In [ ]:
for ticker in TICKERS:
    df = load_ticker(ticker)
    plot_fold_splits(df, ticker)
    
print('All fold split figures saved to results/figures/data/')

## 5. The 25 Technical Indicators

In [ ]:
groups = {
    'Momentum (7)':   ['RSI', 'MACD', 'MACD Signal', 'Stoch %K', 'Stoch %D', 'ROC', 'Williams %R'],
    'Trend (7)':      ['EMA9', 'EMA21', 'EMA50', 'SMA20', 'ADX', 'CCI', 'Parabolic SAR'],
    'Volatility (5)': ['BB Upper', 'BB Lower', 'BB Width', 'ATR', 'Std Dev'],
    'Volume (6)':     ['OBV', 'MFI', 'VWAP', 'Vol SMA', 'Vol ROC', 'CMF'],
}

for group, indicators in groups.items():
    print(f'\n{group}')
    for ind in indicators:
        print(f'  • {ind}')

## 6. Indicator Correlation Heatmap - JKH

High correlations (especially among trend indicators) motivate feature selection.
Including correlated features adds parameters without adding information.

In [ ]:
path = plot_indicator_correlation(df_jkh, ticker='JKH')
Image(path)

## 7. Walk-Forward Fold Statistics

In [ ]:
df = load_ticker('JKH.N0000')
folds = get_folds(df, n_splits=5, val_size=126)

print(f'Total rows : {len(df)}')
print(f'Folds      : {len(folds)}')
print()
print(f'{"Fold":<6} {"Train rows":<12} {"Val rows":<10} {"Train %"}')
print('-' * 40)
for i, fold in enumerate(folds):
    tr = fold['X_train'].shape[0]
    va = fold['X_val'].shape[0]
    print(f'{i+1:<6} {tr:<12} {va:<10} {tr/(tr+va)*100:.1f}%')